Connected to Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["IBM_AML_HiSmall", "IBM_AML_LiSmall", "IBM_AML_HiMedium", "IBM_AML_LiMedium"]
    models = ["XGB", "RF", "SVM"]

print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 19.8 GB available, 36.2% used
Starting batch processing for 4 datasets: IBM_AML_HiSmall, IBM_AML_LiSmall, IBM_AML_HiMedium, IBM_AML_LiMedium

Hyperparameter tuning for datasets 1/4: IBM_AML_HiSmall

Study 'XGB_optimization on IBM_AML_HiSmall dataset' not found.


c:\Users\lambe\Desktop\Final_script\pre_processing.py:99: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Dataset IBM_AML_HiSmall loaded successfully for hyperparameter tuning.
Masks extracted and removed from data variable
Starting hyperparameter optimization for IBM_AML_HiSmall dataset...
Data kept on CPU for NeighborLoader-based training (IBM_AML_HiSmall)
Starting hyperparameter optimization for IBM_AML_HiSmall dataset...
Starting hyperparameter tuning for dataset: IBM_AML_HiSmall


Models:   0%|          | 0/3 [00:00<?, ?model/s][I 2026-05-18 20:44:13,619] A new study created in RDB with name: XGB_optimization on IBM_AML_HiSmall dataset


...

Study 'XGB_optimization on IBM_AML_HiMedium dataset' not found.
Starting optimization for XGB on IBM_AML_HiMedium with 100 trials.


[I 2026-05-19 05:38:59,705] Trial 0 finished with value: 0.08791144373394721 and parameters: {'max_depth': 10, 'Gamma_XGB': 2.3552738895344705, 'n_estimators': 150, 'learning_rate_XGB': 0.010475966639129197, 'colsample_bytree': 0.6019157892116841, 'subsample': 0.733916845526138, 'min_child_weight': 7, 'reg_alpha': 1.471219765672416e-08, 'reg_lambda': 3.32715618968417e-07}. Best is trial 0 with value: 0.08791144373394721.
[I 2026-05-19 05:42:46,103] Trial 1 finished with value: 0.08303591457484508 and parameters: {'max_depth': 14, 'Gamma_XGB': 3.0310167432454564, 'n_estimators': 500, 'learning_rate_XGB': 0.007272455552531321, 'colsample_bytree': 0.8986107470502293, 'subsample': 0.8806334256434325, 'min_child_weight': 1, 'reg_alpha': 0.005604652428395491, 'reg_lambda': 0.0011340128733767346}. Best is trial 0 with value: 0.08791144373394721.
[I 2026-05-19 05:44:08,693] Trial 2 finished with value: 0.07831708090288206 and parameters: {'max_depth': 7, 'Gamma_XGB': 4.849593060524612, 'n_esti

Best val PR-AUC for XGB on IBM_AML_HiMedium: 0.0919
Study 'RF_optimization on IBM_AML_HiMedium dataset' not found.
Starting optimization for RF on IBM_AML_HiMedium with 100 trials.


[I 2026-05-19 09:09:14,814] Trial 0 finished with value: 0.07005665040370315 and parameters: {'n_estimators': 250, 'max_depth': 10, 'min_samples_split': 16, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.07005665040370315.
[I 2026-05-19 09:22:26,917] Trial 1 finished with value: 0.07182751452906405 and parameters: {'n_estimators': 350, 'max_depth': 12, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 1 with value: 0.07182751452906405.
[I 2026-05-19 09:34:25,796] Trial 2 finished with value: 0.07290284045440898 and parameters: {'n_estimators': 300, 'max_depth': 14, 'min_samples_split': 14, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 2 with value: 0.07290284045440898.
[I 2026-05-19 09:50:10,729] Trial 3 finished with value: 0.06926836504708979 and parameters: {'n_estimators': 450, 'max_depth': 9, 'min_samples_split': 13, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 2 with value: 0.07290

MemoryError: Unable to allocate 134. MiB for an array with shape (17601826,) and data type int64

In [ ]:
for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Final evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    # Load best parameters from Optuna databases
    model_parameters = {}
    all_found = True
    for model_name in models:
        study_name = f'{model_name}_optimization on {dataset} dataset'
        storage_url = f'sqlite:///optimization_results_on_{dataset}_{model_name}.db'
        try:
            study = optuna.load_study(study_name=study_name, storage=storage_url)
            model_parameters[model_name] = [study.best_trial.params]
            print(f"Loaded best params for {model_name} (best F1-illicit: {study.best_value:.4f})")
        except KeyError:
            print(f"No completed study found for {model_name} on {dataset}, skipping.")
            all_found = False

    if not model_parameters:
        print(f"No model parameters found for {dataset}. Skipping evaluation.")
        continue

    # Load dataset
    match dataset:
        case "Elliptic":
            from pre_processing import EllipticDataset
            data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
        case "IBM_AML_HiSmall":
            from pre_processing import IBMAMLDataset_HiSmall
            data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
        case "IBM_AML_LiSmall":
            from pre_processing import IBMAMLDataset_LiSmall
            data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
        case "IBM_AML_HiMedium":
            from pre_processing import IBMAMLDataset_HiMedium
            data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
        case "IBM_AML_LiMedium":
            from pre_processing import IBMAMLDataset_LiMedium
            data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
        case "AMLSim":
            from pre_processing import AMLSimDataset
            data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]

    data, masks = extract_and_remove_masks(data)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for evaluation, so all datasets except Elliptic are in this set.

    if dataset not in loader_datasets:
        data = data.to(device)
        print(f"Data moved to device: {device}")
    else:
        print(f"Data kept on CPU for NeighborLoader-based evaluation ({dataset})")

    run_final_evaluation(
        models=list(model_parameters.keys()),
        model_parameters=model_parameters,
        data=data,
        data_for_optimisation=dataset,
        masks=masks
    )

    # Clean up GPU memory between datasets
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f"GPU memory cleared after {dataset}")

    print(f"\n{'='*80}")
    print(f"✓ Successfully completed evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"ALL PROCESSING COMPLETE")
print(f"{'='*80}")

: 

Restarted Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["IBM_AML_HiSmall", "IBM_AML_LiSmall", "IBM_AML_HiMedium", "IBM_AML_LiMedium"]
    models = ["XGB", "RF", "SVM"]

print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 22.0 GB available, 29.3% used
Starting batch processing for 4 datasets: IBM_AML_HiSmall, IBM_AML_LiSmall, IBM_AML_HiMedium, IBM_AML_LiMedium

Hyperparameter tuning for datasets 1/4: IBM_AML_HiSmall

Study 'XGB_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_HiSmall are complete. Skipping to next dataset.

Hyperparameter tuning for datasets 2/4: IBM_AML_LiSmall

Study 'XGB_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_LiSmall are complete. Skipping to n

c:\Users\lambe\Desktop\Final_script\pre_processing.py:599: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Dataset IBM_AML_HiMedium loaded successfully for hyperparameter tuning.
Masks extracted and removed from data variable
Starting hyperparameter optimization for IBM_AML_HiMedium dataset...
Data kept on CPU for NeighborLoader-based training (IBM_AML_HiMedium)
Starting hyperparameter optimization for IBM_AML_HiMedium dataset...
Starting hyperparameter tuning for dataset: IBM_AML_HiMedium


Models:  33%|███▎      | 1/3 [00:00<00:00,  5.11model/s]

Study 'XGB_optimization on IBM_AML_HiMedium dataset' complete: 100/35 trials done.
Study for XGB on IBM_AML_HiMedium already complete. Skipping optimization.


[I 2026-05-19 13:55:54,383] Using an existing study with name 'RF_optimization on IBM_AML_HiMedium dataset' instead of creating a new one.


Study 'RF_optimization on IBM_AML_HiMedium dataset' incomplete: 14/35 trials done, 21 remaining.
Resuming optimization for RF on IBM_AML_HiMedium: 14 trials done, 86 remaining (target: 100).


[I 2026-05-19 14:02:52,156] Trial 15 finished with value: 0.0800080866756686 and parameters: {'n_estimators': 150, 'max_depth': 15, 'min_samples_split': 14, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 15 with value: 0.0800080866756686.
[I 2026-05-19 14:07:38,771] Trial 16 finished with value: 0.07443135400447809 and parameters: {'n_estimators': 100, 'max_depth': 15, 'min_samples_split': 11, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 15 with value: 0.0800080866756686.
[I 2026-05-19 14:16:45,800] Trial 17 finished with value: 0.07797237612104388 and parameters: {'n_estimators': 200, 'max_depth': 15, 'min_samples_split': 14, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 15 with value: 0.0800080866756686.
[I 2026-05-19 14:25:29,665] Trial 18 finished with value: 0.07814563220668547 and parameters: {'n_estimators': 200, 'max_depth': 13, 'min_samples_split': 13, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 15 with value: 0

MemoryError: Unable to allocate 134. MiB for an array with shape (17601826,) and data type float64

In [ ]:
for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Final evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    # Load best parameters from Optuna databases
    model_parameters = {}
    all_found = True
    for model_name in models:
        study_name = f'{model_name}_optimization on {dataset} dataset'
        storage_url = f'sqlite:///optimization_results_on_{dataset}_{model_name}.db'
        try:
            study = optuna.load_study(study_name=study_name, storage=storage_url)
            model_parameters[model_name] = [study.best_trial.params]
            print(f"Loaded best params for {model_name} (best F1-illicit: {study.best_value:.4f})")
        except KeyError:
            print(f"No completed study found for {model_name} on {dataset}, skipping.")
            all_found = False

    if not model_parameters:
        print(f"No model parameters found for {dataset}. Skipping evaluation.")
        continue

    # Load dataset
    match dataset:
        case "Elliptic":
            from pre_processing import EllipticDataset
            data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
        case "IBM_AML_HiSmall":
            from pre_processing import IBMAMLDataset_HiSmall
            data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
        case "IBM_AML_LiSmall":
            from pre_processing import IBMAMLDataset_LiSmall
            data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
        case "IBM_AML_HiMedium":
            from pre_processing import IBMAMLDataset_HiMedium
            data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
        case "IBM_AML_LiMedium":
            from pre_processing import IBMAMLDataset_LiMedium
            data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
        case "AMLSim":
            from pre_processing import AMLSimDataset
            data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]

    data, masks = extract_and_remove_masks(data)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for evaluation, so all datasets except Elliptic are in this set.

    if dataset not in loader_datasets:
        data = data.to(device)
        print(f"Data moved to device: {device}")
    else:
        print(f"Data kept on CPU for NeighborLoader-based evaluation ({dataset})")

    run_final_evaluation(
        models=list(model_parameters.keys()),
        model_parameters=model_parameters,
        data=data,
        data_for_optimisation=dataset,
        masks=masks
    )

    # Clean up GPU memory between datasets
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f"GPU memory cleared after {dataset}")

    print(f"\n{'='*80}")
    print(f"✓ Successfully completed evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"ALL PROCESSING COMPLETE")
print(f"{'='*80}")

: 

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["IBM_AML_HiSmall", "IBM_AML_LiSmall", "IBM_AML_HiMedium", "IBM_AML_LiMedium"]
    models = ["XGB", "RF", "SVM"]

print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

: 

In [ ]:
for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Final evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    # Load best parameters from Optuna databases
    model_parameters = {}
    all_found = True
    for model_name in models:
        study_name = f'{model_name}_optimization on {dataset} dataset'
        storage_url = f'sqlite:///optimization_results_on_{dataset}_{model_name}.db'
        try:
            study = optuna.load_study(study_name=study_name, storage=storage_url)
            model_parameters[model_name] = [study.best_trial.params]
            print(f"Loaded best params for {model_name} (best F1-illicit: {study.best_value:.4f})")
        except KeyError:
            print(f"No completed study found for {model_name} on {dataset}, skipping.")
            all_found = False

    if not model_parameters:
        print(f"No model parameters found for {dataset}. Skipping evaluation.")
        continue

    # Load dataset
    match dataset:
        case "Elliptic":
            from pre_processing import EllipticDataset
            data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
        case "IBM_AML_HiSmall":
            from pre_processing import IBMAMLDataset_HiSmall
            data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
        case "IBM_AML_LiSmall":
            from pre_processing import IBMAMLDataset_LiSmall
            data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
        case "IBM_AML_HiMedium":
            from pre_processing import IBMAMLDataset_HiMedium
            data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
        case "IBM_AML_LiMedium":
            from pre_processing import IBMAMLDataset_LiMedium
            data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
        case "AMLSim":
            from pre_processing import AMLSimDataset
            data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]

    data, masks = extract_and_remove_masks(data)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for evaluation, so all datasets except Elliptic are in this set.

    if dataset not in loader_datasets:
        data = data.to(device)
        print(f"Data moved to device: {device}")
    else:
        print(f"Data kept on CPU for NeighborLoader-based evaluation ({dataset})")

    run_final_evaluation(
        models=list(model_parameters.keys()),
        model_parameters=model_parameters,
        data=data,
        data_for_optimisation=dataset,
        masks=masks
    )

    # Clean up GPU memory between datasets
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f"GPU memory cleared after {dataset}")

    print(f"\n{'='*80}")
    print(f"✓ Successfully completed evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"ALL PROCESSING COMPLETE")
print(f"{'='*80}")

: 

Restarted Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["IBM_AML_HiSmall", "IBM_AML_LiSmall", "IBM_AML_HiMedium", "IBM_AML_LiMedium"]
    models = ["XGB", "RF", "SVM"]

print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 22.3 GB available, 28.3% used
Starting batch processing for 4 datasets: IBM_AML_HiSmall, IBM_AML_LiSmall, IBM_AML_HiMedium, IBM_AML_LiMedium

Hyperparameter tuning for datasets 1/4: IBM_AML_HiSmall

Study 'XGB_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_HiSmall are complete. Skipping to next dataset.

Hyperparameter tuning for datasets 2/4: IBM_AML_LiSmall

Study 'XGB_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_LiSmall are complete. Skipping to n

c:\Users\lambe\Desktop\Final_script\pre_processing.py:599: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Dataset IBM_AML_HiMedium loaded successfully for hyperparameter tuning.
Masks extracted and removed from data variable
Starting hyperparameter optimization for IBM_AML_HiMedium dataset...
Data kept on CPU for NeighborLoader-based training (IBM_AML_HiMedium)
Starting hyperparameter optimization for IBM_AML_HiMedium dataset...
Starting hyperparameter tuning for dataset: IBM_AML_HiMedium


Models:   0%|          | 0/3 [00:00<?, ?model/s][I 2026-05-19 16:27:37,359] Using an existing study with name 'RF_optimization on IBM_AML_HiMedium dataset' instead of creating a new one.


Study 'XGB_optimization on IBM_AML_HiMedium dataset' complete: 100/35 trials done.
Study for XGB on IBM_AML_HiMedium already complete. Skipping optimization.
Study 'RF_optimization on IBM_AML_HiMedium dataset' incomplete: 30/35 trials done, 5 remaining.
Resuming optimization for RF on IBM_AML_HiMedium: 30 trials done, 70 remaining (target: 100).


[I 2026-05-19 16:39:13,847] Trial 32 finished with value: 0.07226939580225168 and parameters: {'n_estimators': 300, 'max_depth': 12, 'min_samples_split': 14, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 21 with value: 0.08135412286515648.
[I 2026-05-19 16:47:58,199] Trial 33 finished with value: 0.07639747899463431 and parameters: {'n_estimators': 200, 'max_depth': 13, 'min_samples_split': 15, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 21 with value: 0.08135412286515648.
[I 2026-05-19 16:54:29,744] Trial 34 finished with value: 0.07813309021052489 and parameters: {'n_estimators': 150, 'max_depth': 13, 'min_samples_split': 9, 'min_samples_leaf': 6, 'max_features': 'sqrt'}. Best is trial 21 with value: 0.08135412286515648.
[I 2026-05-19 17:05:51,048] Trial 35 finished with value: 0.07773936667651575 and parameters: {'n_estimators': 250, 'max_depth': 14, 'min_samples_split': 12, 'min_samples_leaf': 9, 'max_features': 'sqrt'}. Best is trial 21 with value

MemoryError: Unable to allocate 134. MiB for an array with shape (17601826,) and data type int64

Restarted Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["IBM_AML_HiSmall", "IBM_AML_LiSmall", "IBM_AML_HiMedium", "IBM_AML_LiMedium"]
    models = ["XGB", "RF", "SVM"]

print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 23.7 GB available, 23.9% used
Starting batch processing for 4 datasets: IBM_AML_HiSmall, IBM_AML_LiSmall, IBM_AML_HiMedium, IBM_AML_LiMedium

Hyperparameter tuning for datasets 1/4: IBM_AML_HiSmall

Study 'XGB_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_HiSmall are complete. Skipping to next dataset.

Hyperparameter tuning for datasets 2/4: IBM_AML_LiSmall

Study 'XGB_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_LiSmall are complete. Skipping to n

c:\Users\lambe\Desktop\Final_script\pre_processing.py:599: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Dataset IBM_AML_HiMedium loaded successfully for hyperparameter tuning.
Masks extracted and removed from data variable
Starting hyperparameter optimization for IBM_AML_HiMedium dataset...
Data kept on CPU for NeighborLoader-based training (IBM_AML_HiMedium)
Starting hyperparameter optimization for IBM_AML_HiMedium dataset...
Starting hyperparameter tuning for dataset: IBM_AML_HiMedium


Models:   0%|          | 0/3 [00:00<?, ?model/s][I 2026-05-19 17:39:48,518] A new study created in RDB with name: SVM_optimization on IBM_AML_HiMedium dataset


Study 'XGB_optimization on IBM_AML_HiMedium dataset' complete: 100/35 trials done.
Study for XGB on IBM_AML_HiMedium already complete. Skipping optimization.
Study 'RF_optimization on IBM_AML_HiMedium dataset' complete: 37/35 trials done.
Study for RF on IBM_AML_HiMedium already complete. Skipping optimization.
Study 'SVM_optimization on IBM_AML_HiMedium dataset' not found.
Starting optimization for SVM on IBM_AML_HiMedium with 100 trials.


[I 2026-05-19 17:43:07,255] Trial 0 finished with value: 0.010330747360725078 and parameters: {'C': 0.06522512528818461}. Best is trial 0 with value: 0.010330747360725078.
[I 2026-05-19 17:51:19,668] Trial 1 finished with value: 0.011805741292703106 and parameters: {'C': 74.48087609501407}. Best is trial 1 with value: 0.011805741292703106.
[I 2026-05-19 17:55:16,485] Trial 2 finished with value: 0.004571433175559375 and parameters: {'C': 2.5503097276177633}. Best is trial 1 with value: 0.011805741292703106.
[I 2026-05-19 18:00:27,235] Trial 3 finished with value: 0.011138905013581108 and parameters: {'C': 0.3942389553889887}. Best is trial 1 with value: 0.011805741292703106.
[I 2026-05-19 18:05:53,271] Trial 4 finished with value: 0.011078420012572462 and parameters: {'C': 0.449233526568865}. Best is trial 1 with value: 0.011805741292703106.
[I 2026-05-19 18:11:08,467] Trial 5 finished with value: 0.01196473397418936 and parameters: {'C': 0.4162589816978857}. Best is trial 5 with value

Restarted Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["IBM_AML_HiSmall", "IBM_AML_LiSmall", "IBM_AML_HiMedium", "IBM_AML_LiMedium"]
    models = ["XGB", "RF", "SVM"]

print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 21.5 GB available, 30.9% used
Starting batch processing for 4 datasets: IBM_AML_HiSmall, IBM_AML_LiSmall, IBM_AML_HiMedium, IBM_AML_LiMedium

Hyperparameter tuning for datasets 1/4: IBM_AML_HiSmall

Study 'XGB_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_HiSmall are complete. Skipping to next dataset.

Hyperparameter tuning for datasets 2/4: IBM_AML_LiSmall

Study 'XGB_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_LiSmall are complete. Skipping to n

c:\Users\lambe\Desktop\Final_script\pre_processing.py:434: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Dataset IBM_AML_LiMedium loaded successfully for hyperparameter tuning.
Masks extracted and removed from data variable
Starting hyperparameter optimization for IBM_AML_LiMedium dataset...
Data kept on CPU for NeighborLoader-based training (IBM_AML_LiMedium)
Starting hyperparameter optimization for IBM_AML_LiMedium dataset...
Starting hyperparameter tuning for dataset: IBM_AML_LiMedium


Models:   0%|          | 0/3 [00:00<?, ?model/s][I 2026-05-19 21:34:04,201] A new study created in RDB with name: XGB_optimization on IBM_AML_LiMedium dataset


Study 'XGB_optimization on IBM_AML_LiMedium dataset' not found.
Starting optimization for XGB on IBM_AML_LiMedium with 100 trials.


[I 2026-05-19 21:34:32,987] Trial 0 finished with value: 0.009153263315959928 and parameters: {'max_depth': 14, 'Gamma_XGB': 3.693105843662561, 'n_estimators': 50, 'learning_rate_XGB': 0.0010425853272619749, 'colsample_bytree': 0.5197062658930255, 'subsample': 0.8314091740213492, 'min_child_weight': 10, 'reg_alpha': 0.009876102992354094, 'reg_lambda': 0.015219625909620428}. Best is trial 0 with value: 0.009153263315959928.
[I 2026-05-19 21:35:10,440] Trial 1 finished with value: 0.010907679273672092 and parameters: {'max_depth': 6, 'Gamma_XGB': 3.0248015688843366, 'n_estimators': 150, 'learning_rate_XGB': 0.05420105127671779, 'colsample_bytree': 0.5342757581313795, 'subsample': 0.9177564589390029, 'min_child_weight': 4, 'reg_alpha': 0.20241115736514134, 'reg_lambda': 1.2166319375666484e-06}. Best is trial 1 with value: 0.010907679273672092.
[I 2026-05-19 21:37:17,847] Trial 2 finished with value: 0.01031016758601896 and parameters: {'max_depth': 8, 'Gamma_XGB': 4.524774086667833, 'n_es

Best val PR-AUC for XGB on IBM_AML_LiMedium: 0.0116
Study 'RF_optimization on IBM_AML_LiMedium dataset' not found.
Starting optimization for RF on IBM_AML_LiMedium with 100 trials.


[I 2026-05-19 23:16:56,134] Trial 0 finished with value: 0.00889337513997182 and parameters: {'n_estimators': 200, 'max_depth': 11, 'min_samples_split': 13, 'min_samples_leaf': 8, 'max_features': 'log2'}. Best is trial 0 with value: 0.00889337513997182.
[I 2026-05-19 23:23:13,848] Trial 1 finished with value: 0.008347169939688674 and parameters: {'n_estimators': 250, 'max_depth': 6, 'min_samples_split': 9, 'min_samples_leaf': 9, 'max_features': 'log2'}. Best is trial 0 with value: 0.00889337513997182.
[I 2026-05-19 23:30:38,002] Trial 2 finished with value: 0.008208456174675575 and parameters: {'n_estimators': 250, 'max_depth': 8, 'min_samples_split': 20, 'min_samples_leaf': 7, 'max_features': 'log2'}. Best is trial 0 with value: 0.00889337513997182.
[I 2026-05-19 23:42:57,269] Trial 3 finished with value: 0.00855375483387676 and parameters: {'n_estimators': 400, 'max_depth': 9, 'min_samples_split': 12, 'min_samples_leaf': 7, 'max_features': 'log2'}. Best is trial 0 with value: 0.00889

MemoryError: Unable to allocate 132. MiB for an array with shape (17251279,) and data type int64

Connected to Home_PC_cuda (Python 3.10.19)

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["IBM_AML_HiSmall", "IBM_AML_LiSmall", "IBM_AML_HiMedium", "IBM_AML_LiMedium"]
    models = ["XGB", "RF", "SVM"]

print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 16.5 GB available, 46.8% used
Starting batch processing for 4 datasets: IBM_AML_HiSmall, IBM_AML_LiSmall, IBM_AML_HiMedium, IBM_AML_LiMedium

Hyperparameter tuning for datasets 1/4: IBM_AML_HiSmall

Study 'XGB_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_HiSmall are complete. Skipping to next dataset.

Hyperparameter tuning for datasets 2/4: IBM_AML_LiSmall

Study 'XGB_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_LiSmall are complete. Skipping to n

c:\Users\lambe\Desktop\Final_script\pre_processing.py:434: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Dataset IBM_AML_LiMedium loaded successfully for hyperparameter tuning.
Masks extracted and removed from data variable
Starting hyperparameter optimization for IBM_AML_LiMedium dataset...
Data kept on CPU for NeighborLoader-based training (IBM_AML_LiMedium)
Starting hyperparameter optimization for IBM_AML_LiMedium dataset...
Starting hyperparameter tuning for dataset: IBM_AML_LiMedium


Models:   0%|          | 0/3 [00:00<?, ?model/s][I 2026-05-20 08:17:24,560] Using an existing study with name 'RF_optimization on IBM_AML_LiMedium dataset' instead of creating a new one.


Study 'XGB_optimization on IBM_AML_LiMedium dataset' complete: 100/35 trials done.
Study for XGB on IBM_AML_LiMedium already complete. Skipping optimization.
Study 'RF_optimization on IBM_AML_LiMedium dataset' incomplete: 23/35 trials done, 12 remaining.
Resuming optimization for RF on IBM_AML_LiMedium: 23 trials done, 77 remaining (target: 100).


[I 2026-05-20 08:19:39,827] Trial 24 finished with value: 0.008682630061335959 and parameters: {'n_estimators': 50, 'max_depth': 9, 'min_samples_split': 13, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 22 with value: 0.009352596310916566.
[I 2026-05-20 08:21:57,731] Trial 25 finished with value: 0.00865306649142785 and parameters: {'n_estimators': 50, 'max_depth': 12, 'min_samples_split': 15, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 22 with value: 0.009352596310916566.
[I 2026-05-20 08:25:40,290] Trial 26 finished with value: 0.009138995424733143 and parameters: {'n_estimators': 100, 'max_depth': 9, 'min_samples_split': 13, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 22 with value: 0.009352596310916566.
[W 2026-05-20 08:25:49,780] Trial 27 failed with parameters: {'n_estimators': 200, 'max_depth': 9, 'min_samples_split': 17, 'min_samples_leaf': 2, 'max_features': 'sqrt'} because of the following error: MemoryError((17251279,), dty

MemoryError: Unable to allocate 132. MiB for an array with shape (17251279,) and data type int64

Restarted Home_PC_cuda (Python 3.10.19)

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["IBM_AML_HiSmall", "IBM_AML_LiSmall", "IBM_AML_HiMedium", "IBM_AML_LiMedium"]
    models = ["XGB", "RF", "SVM"]

print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 23.8 GB available, 23.5% used
Starting batch processing for 4 datasets: IBM_AML_HiSmall, IBM_AML_LiSmall, IBM_AML_HiMedium, IBM_AML_LiMedium

Hyperparameter tuning for datasets 1/4: IBM_AML_HiSmall

Study 'XGB_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_HiSmall are complete. Skipping to next dataset.

Hyperparameter tuning for datasets 2/4: IBM_AML_LiSmall

Study 'XGB_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_LiSmall are complete. Skipping to n

c:\Users\lambe\Desktop\Final_script\pre_processing.py:434: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Dataset IBM_AML_LiMedium loaded successfully for hyperparameter tuning.
Masks extracted and removed from data variable
Starting hyperparameter optimization for IBM_AML_LiMedium dataset...
Data kept on CPU for NeighborLoader-based training (IBM_AML_LiMedium)
Starting hyperparameter optimization for IBM_AML_LiMedium dataset...
Starting hyperparameter tuning for dataset: IBM_AML_LiMedium


Models:   0%|          | 0/3 [00:00<?, ?model/s][I 2026-05-20 08:31:48,532] Using an existing study with name 'RF_optimization on IBM_AML_LiMedium dataset' instead of creating a new one.


Study 'XGB_optimization on IBM_AML_LiMedium dataset' complete: 100/35 trials done.
Study for XGB on IBM_AML_LiMedium already complete. Skipping optimization.
Study 'RF_optimization on IBM_AML_LiMedium dataset' incomplete: 26/35 trials done, 9 remaining.
Resuming optimization for RF on IBM_AML_LiMedium: 26 trials done, 74 remaining (target: 100).


[I 2026-05-20 08:38:59,340] Trial 28 finished with value: 0.008431653594123822 and parameters: {'n_estimators': 200, 'max_depth': 9, 'min_samples_split': 17, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 22 with value: 0.009352596310916566.
[I 2026-05-20 08:42:10,283] Trial 29 finished with value: 0.008532423219632318 and parameters: {'n_estimators': 100, 'max_depth': 7, 'min_samples_split': 13, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 22 with value: 0.009352596310916566.
[I 2026-05-20 08:44:38,612] Trial 30 finished with value: 0.009019491208560442 and parameters: {'n_estimators': 50, 'max_depth': 14, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 22 with value: 0.009352596310916566.
[I 2026-05-20 08:51:03,828] Trial 31 finished with value: 0.008464622405881042 and parameters: {'n_estimators': 200, 'max_depth': 8, 'min_samples_split': 13, 'min_samples_leaf': 6, 'max_features': 'sqrt'}. Best is trial 22 with va

Connected to Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["IBM_AML_HiSmall", "IBM_AML_LiSmall", "IBM_AML_HiMedium", "IBM_AML_LiMedium"]
    models = ["XGB", "RF", "SVM"]

print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 15.8 GB available, 49.4% used
Starting batch processing for 4 datasets: IBM_AML_HiSmall, IBM_AML_LiSmall, IBM_AML_HiMedium, IBM_AML_LiMedium

Hyperparameter tuning for datasets 1/4: IBM_AML_HiSmall

Study 'XGB_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_HiSmall are complete. Skipping to next dataset.

Hyperparameter tuning for datasets 2/4: IBM_AML_LiSmall

Study 'XGB_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_LiSmall are complete. Skipping to n

c:\Users\lambe\Desktop\Final_script\pre_processing.py:434: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Dataset IBM_AML_LiMedium loaded successfully for hyperparameter tuning.
Masks extracted and removed from data variable
Starting hyperparameter optimization for IBM_AML_LiMedium dataset...
Data kept on CPU for NeighborLoader-based training (IBM_AML_LiMedium)
Starting hyperparameter optimization for IBM_AML_LiMedium dataset...
Starting hyperparameter tuning for dataset: IBM_AML_LiMedium


Models:   0%|          | 0/3 [00:00<?, ?model/s][I 2026-05-20 09:47:21,085] A new study created in RDB with name: SVM_optimization on IBM_AML_LiMedium dataset


Study 'XGB_optimization on IBM_AML_LiMedium dataset' complete: 100/35 trials done.
Study for XGB on IBM_AML_LiMedium already complete. Skipping optimization.
Study 'RF_optimization on IBM_AML_LiMedium dataset' complete: 37/35 trials done.
Study for RF on IBM_AML_LiMedium already complete. Skipping optimization.
Study 'SVM_optimization on IBM_AML_LiMedium dataset' not found.
Starting optimization for SVM on IBM_AML_LiMedium with 100 trials.


[I 2026-05-20 09:50:45,944] Trial 0 finished with value: 0.003711545387594996 and parameters: {'C': 1.7238522978946949}. Best is trial 0 with value: 0.003711545387594996.
[I 2026-05-20 09:58:51,403] Trial 1 finished with value: 0.0036333852897757852 and parameters: {'C': 51.499744258683265}. Best is trial 0 with value: 0.003711545387594996.
[I 2026-05-20 10:04:25,766] Trial 2 finished with value: 0.0038202342446162986 and parameters: {'C': 0.04629449392666828}. Best is trial 2 with value: 0.0038202342446162986.
[I 2026-05-20 10:08:05,419] Trial 3 finished with value: 0.0034006461027856648 and parameters: {'C': 0.01684825822086615}. Best is trial 2 with value: 0.0038202342446162986.
[I 2026-05-20 10:13:21,973] Trial 4 finished with value: 0.00354758980166699 and parameters: {'C': 2.1116038568182454}. Best is trial 2 with value: 0.0038202342446162986.
[I 2026-05-20 10:17:25,564] Trial 5 finished with value: 0.002918875127511243 and parameters: {'C': 3.8790829791215677}. Best is trial 2 w

Restarted Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["IBM_AML_HiSmall", "IBM_AML_LiSmall"]
    models = ["XGB", "RF", "SVM"]

print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 20.3 GB available, 34.8% used
Starting batch processing for 2 datasets: IBM_AML_HiSmall, IBM_AML_LiSmall

Hyperparameter tuning for datasets 1/2: IBM_AML_HiSmall

Study 'XGB_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_HiSmall are complete. Skipping to next dataset.

Hyperparameter tuning for datasets 2/2: IBM_AML_LiSmall

Study 'XGB_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimization on IBM_AML_LiSmall dataset' complete: 100/35 trials done.
All studies for IBM_AML_LiSmall are complete. Skipping to next dataset.

HYPERPARAMETER TUNING 

In [ ]:
for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Final evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    # Load best parameters from Optuna databases
    model_parameters = {}
    all_found = True
    for model_name in models:
        study_name = f'{model_name}_optimization on {dataset} dataset'
        storage_url = f'sqlite:///optimization_results_on_{dataset}_{model_name}.db'
        try:
            study = optuna.load_study(study_name=study_name, storage=storage_url)
            model_parameters[model_name] = [study.best_trial.params]
            print(f"Loaded best params for {model_name} (best F1-illicit: {study.best_value:.4f})")
        except KeyError:
            print(f"No completed study found for {model_name} on {dataset}, skipping.")
            all_found = False

    if not model_parameters:
        print(f"No model parameters found for {dataset}. Skipping evaluation.")
        continue

    # Load dataset
    match dataset:
        case "Elliptic":
            from pre_processing import EllipticDataset
            data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
        case "IBM_AML_HiSmall":
            from pre_processing import IBMAMLDataset_HiSmall
            data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
        case "IBM_AML_LiSmall":
            from pre_processing import IBMAMLDataset_LiSmall
            data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
        case "IBM_AML_HiMedium":
            from pre_processing import IBMAMLDataset_HiMedium
            data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
        case "IBM_AML_LiMedium":
            from pre_processing import IBMAMLDataset_LiMedium
            data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
        case "AMLSim":
            from pre_processing import AMLSimDataset
            data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]

    data, masks = extract_and_remove_masks(data)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for evaluation, so all datasets except Elliptic are in this set.

    if dataset not in loader_datasets:
        data = data.to(device)
        print(f"Data moved to device: {device}")
    else:
        print(f"Data kept on CPU for NeighborLoader-based evaluation ({dataset})")

    run_final_evaluation(
        models=list(model_parameters.keys()),
        model_parameters=model_parameters,
        data=data,
        data_for_optimisation=dataset,
        masks=masks
    )

    # Clean up GPU memory between datasets
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f"GPU memory cleared after {dataset}")

    print(f"\n{'='*80}")
    print(f"✓ Successfully completed evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"ALL PROCESSING COMPLETE")
print(f"{'='*80}")


Final evaluation for dataset 1/2: IBM_AML_HiSmall

Loaded best params for XGB (best F1-illicit: 0.0687)
Loaded best params for RF (best F1-illicit: 0.0591)
Loaded best params for SVM (best F1-illicit: 0.0111)


c:\Users\lambe\Desktop\Final_script\pre_processing.py:99: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Data kept on CPU for NeighborLoader-based evaluation (IBM_AML_HiSmall)

Starting FINAL EVALUATION for IBM_AML_HiSmall dataset...
Evaluating XGB on IBM_AML_HiSmall...
Finding optimal EVALUATION batch size for XGB...
Searching for optimal evaluation batch size...
Running in EVALUATION mode: will use up to 80% of GPU memory
GPU memory limit: 12.79 GB
Testing batch size: 16384
Batch size 16384 failed with error: AttributeError: 'XGBClassifier' object has no attribute 'to'
Fallback testing batch size: 16384
Batch size 16384 failed with error: AttributeError: 'XGBClassifier' object has no attribute 'to'
Fallback testing batch size: 8192
Batch size 8192 failed with error: AttributeError: 'XGBClassifier' object has no attribute 'to'
Fallback testing batch size: 4096
Batch size 4096 failed with error: AttributeError: 'XGBClassifier' object has no attribute 'to'
Fallback testing batch size: 2048
Batch size 2048 failed with error: AttributeError: 'XGBClassifier' object has no attribute 'to'
Fallb

Restarted Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["IBM_AML_HiSmall", "IBM_AML_LiSmall"]
    models = ["XGB", "RF", "SVM"]

print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 19.9 GB available, 36.0% used
Memory limit: none detected (using host RAM)
--- cgroup diagnostic (probe returned None) ---
  (failed to read /proc/self/cgroup: [Errno 2] No such file or directory: '/proc/self/cgroup')
  (failed to read /proc/self/mountinfo: [Errno 2] No such file or directory: '/proc/self/mountinfo')
ls /sys/fs/cgroup/memory: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup/memory')
ls /sys/fs/cgroup: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup')
PBS/MEM env vars: {}
--- end diagnostic ---
Starting batch processing for 2 datasets: IBM_AML_HiSmall, IBM_AML_LiSmall

Hyperparameter tuning for datasets 1/2: IBM_AML_HiSmall

Study 'XGB_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimi

In [ ]:
for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Final evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    # Load best parameters from Optuna databases
    model_parameters = {}
    all_found = True
    for model_name in models:
        study_name = f'{model_name}_optimization on {dataset} dataset'
        storage_url = f'sqlite:///optimization_results_on_{dataset}_{model_name}.db'
        try:
            study = optuna.load_study(study_name=study_name, storage=storage_url)
            model_parameters[model_name] = [study.best_trial.params]
            print(f"Loaded best params for {model_name} (best F1-illicit: {study.best_value:.4f})")
        except KeyError:
            print(f"No completed study found for {model_name} on {dataset}, skipping.")
            all_found = False

    if not model_parameters:
        print(f"No model parameters found for {dataset}. Skipping evaluation.")
        continue

    # Load dataset
    match dataset:
        case "Elliptic":
            from pre_processing import EllipticDataset
            data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
        case "IBM_AML_HiSmall":
            from pre_processing import IBMAMLDataset_HiSmall
            data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
        case "IBM_AML_LiSmall":
            from pre_processing import IBMAMLDataset_LiSmall
            data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
        case "IBM_AML_HiMedium":
            from pre_processing import IBMAMLDataset_HiMedium
            data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
        case "IBM_AML_LiMedium":
            from pre_processing import IBMAMLDataset_LiMedium
            data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
        case "AMLSim":
            from pre_processing import AMLSimDataset
            data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]

    data, masks = extract_and_remove_masks(data)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for evaluation, so all datasets except Elliptic are in this set.

    if dataset not in loader_datasets:
        data = data.to(device)
        print(f"Data moved to device: {device}")
    else:
        print(f"Data kept on CPU for NeighborLoader-based evaluation ({dataset})")

    run_final_evaluation(
        models=list(model_parameters.keys()),
        model_parameters=model_parameters,
        data=data,
        data_for_optimisation=dataset,
        masks=masks
    )

    # Clean up GPU memory between datasets
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f"GPU memory cleared after {dataset}")

    print(f"\n{'='*80}")
    print(f"✓ Successfully completed evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"ALL PROCESSING COMPLETE")
print(f"{'='*80}")


Final evaluation for dataset 1/2: IBM_AML_HiSmall

Loaded best params for XGB (best F1-illicit: 0.0687)
Loaded best params for RF (best F1-illicit: 0.0591)
Loaded best params for SVM (best F1-illicit: 0.0111)


c:\Users\lambe\Desktop\Final_script\pre_processing.py:99: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Data kept on CPU for NeighborLoader-based evaluation (IBM_AML_HiSmall)

Starting FINAL EVALUATION for IBM_AML_HiSmall dataset...
Evaluating XGB on IBM_AML_HiSmall...
Finding optimal EVALUATION batch size for XGB...
Searching for optimal evaluation batch size...
Running in EVALUATION mode: will use up to 80% of GPU memory
GPU memory limit: 12.79 GB
Testing batch size: 16384
Batch size 16384 failed with error: AttributeError: 'XGBClassifier' object has no attribute 'to'
Fallback testing batch size: 16384
Batch size 16384 failed with error: AttributeError: 'XGBClassifier' object has no attribute 'to'
Fallback testing batch size: 8192
Batch size 8192 failed with error: AttributeError: 'XGBClassifier' object has no attribute 'to'
Fallback testing batch size: 4096
Batch size 4096 failed with error: AttributeError: 'XGBClassifier' object has no attribute 'to'
Fallback testing batch size: 2048
Batch size 2048 failed with error: AttributeError: 'XGBClassifier' object has no attribute 'to'
Fallb

Restarted Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["IBM_AML_HiSmall", "IBM_AML_LiSmall"]
    models = ["XGB", "RF", "SVM"]

print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 19.6 GB available, 36.9% used
Memory limit: none detected (using host RAM)
--- cgroup diagnostic (probe returned None) ---
  (failed to read /proc/self/cgroup: [Errno 2] No such file or directory: '/proc/self/cgroup')
  (failed to read /proc/self/mountinfo: [Errno 2] No such file or directory: '/proc/self/mountinfo')
ls /sys/fs/cgroup/memory: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup/memory')
ls /sys/fs/cgroup: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup')
PBS/MEM env vars: {}
--- end diagnostic ---
Starting batch processing for 2 datasets: IBM_AML_HiSmall, IBM_AML_LiSmall

Hyperparameter tuning for datasets 1/2: IBM_AML_HiSmall

Study 'XGB_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'RF_optimization on IBM_AML_HiSmall dataset' complete: 100/35 trials done.
Study 'SVM_optimi

In [ ]:
for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Final evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    # Load best parameters from Optuna databases
    model_parameters = {}
    all_found = True
    for model_name in models:
        study_name = f'{model_name}_optimization on {dataset} dataset'
        storage_url = f'sqlite:///optimization_results_on_{dataset}_{model_name}.db'
        try:
            study = optuna.load_study(study_name=study_name, storage=storage_url)
            model_parameters[model_name] = [study.best_trial.params]
            print(f"Loaded best params for {model_name} (best F1-illicit: {study.best_value:.4f})")
        except KeyError:
            print(f"No completed study found for {model_name} on {dataset}, skipping.")
            all_found = False

    if not model_parameters:
        print(f"No model parameters found for {dataset}. Skipping evaluation.")
        continue

    # Load dataset
    match dataset:
        case "Elliptic":
            from pre_processing import EllipticDataset
            data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
        case "IBM_AML_HiSmall":
            from pre_processing import IBMAMLDataset_HiSmall
            data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
        case "IBM_AML_LiSmall":
            from pre_processing import IBMAMLDataset_LiSmall
            data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
        case "IBM_AML_HiMedium":
            from pre_processing import IBMAMLDataset_HiMedium
            data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
        case "IBM_AML_LiMedium":
            from pre_processing import IBMAMLDataset_LiMedium
            data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
        case "AMLSim":
            from pre_processing import AMLSimDataset
            data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]

    data, masks = extract_and_remove_masks(data)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for evaluation, so all datasets except Elliptic are in this set.

    if dataset not in loader_datasets:
        data = data.to(device)
        print(f"Data moved to device: {device}")
    else:
        print(f"Data kept on CPU for NeighborLoader-based evaluation ({dataset})")

    run_final_evaluation(
        models=list(model_parameters.keys()),
        model_parameters=model_parameters,
        data=data,
        data_for_optimisation=dataset,
        masks=masks
    )

    # Clean up GPU memory between datasets
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f"GPU memory cleared after {dataset}")

    print(f"\n{'='*80}")
    print(f"✓ Successfully completed evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"ALL PROCESSING COMPLETE")
print(f"{'='*80}")


Final evaluation for dataset 1/2: IBM_AML_HiSmall

Loaded best params for XGB (best F1-illicit: 0.0687)
Loaded best params for RF (best F1-illicit: 0.0591)
Loaded best params for SVM (best F1-illicit: 0.0111)


c:\Users\lambe\Desktop\Final_script\pre_processing.py:99: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Data kept on CPU for NeighborLoader-based evaluation (IBM_AML_HiSmall)

Starting FINAL EVALUATION for IBM_AML_HiSmall dataset...
Evaluating XGB on IBM_AML_HiSmall...
  > Using FULL BATCH for XGB on IBM_AML_HiSmall (no NeighborLoader)
  > Run 1/10 (Seed 2300741964)
Saved metrics to results/IBM_AML_HiSmall/pkl_files/XGB_run_1_pr_data_local-18228.pkl
Saved metrics to results/IBM_AML_HiSmall/pkl_files/XGB_run_1_metrics_local-18228.pkl
  > Run 2/10 (Seed 3507820380)
Saved metrics to results/IBM_AML_HiSmall/pkl_files/XGB_run_2_pr_data_local-18228.pkl
Saved metrics to results/IBM_AML_HiSmall/pkl_files/XGB_run_2_metrics_local-18228.pkl
  > Run 3/10 (Seed 871889682)
Saved metrics to results/IBM_AML_HiSmall/pkl_files/XGB_run_3_pr_data_local-18228.pkl
Saved metrics to results/IBM_AML_HiSmall/pkl_files/XGB_run_3_metrics_local-18228.pkl
  > Run 4/10 (Seed 764187089)
Saved metrics to results/IBM_AML_HiSmall/pkl_files/XGB_run_4_pr_data_local-18228.pkl
Saved metrics to results/IBM_AML_HiSmall/pkl_file

c:\Users\lambe\Desktop\Final_script\pre_processing.py:269: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Data kept on CPU for NeighborLoader-based evaluation (IBM_AML_LiSmall)

Starting FINAL EVALUATION for IBM_AML_LiSmall dataset...
Evaluating XGB on IBM_AML_LiSmall...
  > Using FULL BATCH for XGB on IBM_AML_LiSmall (no NeighborLoader)
  > Run 1/10 (Seed 1422211318)
Saved metrics to results/IBM_AML_LiSmall/pkl_files/XGB_run_1_pr_data_local-18228.pkl
Saved metrics to results/IBM_AML_LiSmall/pkl_files/XGB_run_1_metrics_local-18228.pkl
  > Run 2/10 (Seed 3111120450)
Saved metrics to results/IBM_AML_LiSmall/pkl_files/XGB_run_2_pr_data_local-18228.pkl
Saved metrics to results/IBM_AML_LiSmall/pkl_files/XGB_run_2_metrics_local-18228.pkl
  > Run 3/10 (Seed 2271775649)
Saved metrics to results/IBM_AML_LiSmall/pkl_files/XGB_run_3_pr_data_local-18228.pkl
Saved metrics to results/IBM_AML_LiSmall/pkl_files/XGB_run_3_metrics_local-18228.pkl
  > Run 4/10 (Seed 3173748605)
Saved metrics to results/IBM_AML_LiSmall/pkl_files/XGB_run_4_pr_data_local-18228.pkl
Saved metrics to results/IBM_AML_LiSmall/pkl_fi

Restarted Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["AMLSim"]
    models = [ "MLP"]

print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 19.6 GB available, 36.9% used
Memory limit: none detected (using host RAM)
--- cgroup diagnostic (probe returned None) ---
  (failed to read /proc/self/cgroup: [Errno 2] No such file or directory: '/proc/self/cgroup')
  (failed to read /proc/self/mountinfo: [Errno 2] No such file or directory: '/proc/self/mountinfo')
ls /sys/fs/cgroup/memory: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup/memory')
ls /sys/fs/cgroup: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup')
PBS/MEM env vars: {}
--- end diagnostic ---
Starting batch processing for 1 datasets: AMLSim

Hyperparameter tuning for datasets 1/1: AMLSim



c:\Users\lambe\Desktop\Final_script\pre_processing.py:766: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Study 'MLP_optimization on AMLSim dataset' not found.
Dataset AMLSim loaded successfully for hyperparameter tuning.
Masks extracted and removed from data variable
Starting hyperparameter optimization for AMLSim dataset...
Data kept on CPU for NeighborLoader-based training (AMLSim)
Starting hyperparameter optimization for AMLSim dataset...
Starting hyperparameter tuning for dataset: AMLSim


Models:   0%|          | 0/1 [00:00<?, ?model/s]

Study 'MLP_optimization on AMLSim dataset' not found.
Searching for optimal tuning batch size...
Running in TUNING mode: will use up to 70% of GPU memory
GPU memory limit: 11.19 GB
Fallback testing batch size: 16384


[I 2026-05-20 13:42:38,402] A new study created in RDB with name: MLP_optimization on AMLSim dataset


Optimal tuning batch size found: 15564
Saved tuning batch size 15564 for AMLSim_MLP
Starting optimization for MLP on AMLSim with 150 trials.


[W 2026-05-20 13:42:48,962] Trial 0 failed with parameters: {'hidden_units': 96, 'dropout_1': 0.04146229795938265, 'dropout_2': 0.1904460691430011, 'learning_rate': 0.0011188284588571338, 'weight_decay': 0.0013925483487829298, 'gamma_focal': 4.447479758987205, 'n_epochs': 108} because of the following error: RuntimeError('DataLoader worker (pid(s) 39256, 24428) exited unexpectedly').
Traceback (most recent call last):
  File "c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\torch\utils\data\dataloader.py", line 1131, in _try_get_data
    data = self._data_queue.get(timeout=timeout)
  File "c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\queue.py", line 179, in get
    raise Empty
_queue.Empty

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\optuna\study\_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "c:\Users\lambe\Des

RuntimeError: DataLoader worker (pid(s) 39256, 24428) exited unexpectedly

In [ ]:
for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Final evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    # Load best parameters from Optuna databases
    model_parameters = {}
    all_found = True
    for model_name in models:
        study_name = f'{model_name}_optimization on {dataset} dataset'
        storage_url = f'sqlite:///optimization_results_on_{dataset}_{model_name}.db'
        try:
            study = optuna.load_study(study_name=study_name, storage=storage_url)
            model_parameters[model_name] = [study.best_trial.params]
            print(f"Loaded best params for {model_name} (best F1-illicit: {study.best_value:.4f})")
        except KeyError:
            print(f"No completed study found for {model_name} on {dataset}, skipping.")
            all_found = False

    if not model_parameters:
        print(f"No model parameters found for {dataset}. Skipping evaluation.")
        continue

    # Load dataset
    match dataset:
        case "Elliptic":
            from pre_processing import EllipticDataset
            data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
        case "IBM_AML_HiSmall":
            from pre_processing import IBMAMLDataset_HiSmall
            data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
        case "IBM_AML_LiSmall":
            from pre_processing import IBMAMLDataset_LiSmall
            data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
        case "IBM_AML_HiMedium":
            from pre_processing import IBMAMLDataset_HiMedium
            data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
        case "IBM_AML_LiMedium":
            from pre_processing import IBMAMLDataset_LiMedium
            data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
        case "AMLSim":
            from pre_processing import AMLSimDataset
            data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]

    data, masks = extract_and_remove_masks(data)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for evaluation, so all datasets except Elliptic are in this set.

    if dataset not in loader_datasets:
        data = data.to(device)
        print(f"Data moved to device: {device}")
    else:
        print(f"Data kept on CPU for NeighborLoader-based evaluation ({dataset})")

    run_final_evaluation(
        models=list(model_parameters.keys()),
        model_parameters=model_parameters,
        data=data,
        data_for_optimisation=dataset,
        masks=masks
    )

    # Clean up GPU memory between datasets
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f"GPU memory cleared after {dataset}")

    print(f"\n{'='*80}")
    print(f"✓ Successfully completed evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"ALL PROCESSING COMPLETE")
print(f"{'='*80}")

: 

Restarted Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["AMLSim"]
    models = [ "MLP"]

print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 19.4 GB available, 37.5% used
Memory limit: none detected (using host RAM)
--- cgroup diagnostic (probe returned None) ---
  (failed to read /proc/self/cgroup: [Errno 2] No such file or directory: '/proc/self/cgroup')
  (failed to read /proc/self/mountinfo: [Errno 2] No such file or directory: '/proc/self/mountinfo')
ls /sys/fs/cgroup/memory: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup/memory')
ls /sys/fs/cgroup: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup')
PBS/MEM env vars: {}
--- end diagnostic ---
Starting batch processing for 1 datasets: AMLSim

Hyperparameter tuning for datasets 1/1: AMLSim



c:\Users\lambe\Desktop\Final_script\pre_processing.py:766: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Study 'MLP_optimization on AMLSim dataset' incomplete: 3/150 trials done, 147 remaining.
Dataset AMLSim loaded successfully for hyperparameter tuning.
Masks extracted and removed from data variable
Starting hyperparameter optimization for AMLSim dataset...
Data kept on CPU for NeighborLoader-based training (AMLSim)
Starting hyperparameter optimization for AMLSim dataset...
Starting hyperparameter tuning for dataset: AMLSim


Models:   0%|          | 0/1 [00:00<?, ?model/s]

Study 'MLP_optimization on AMLSim dataset' incomplete: 3/150 trials done, 147 remaining.
Searching for optimal tuning batch size...
Running in TUNING mode: will use up to 70% of GPU memory
GPU memory limit: 11.19 GB
Fallback testing batch size: 16384


[I 2026-05-20 13:46:19,032] Using an existing study with name 'MLP_optimization on AMLSim dataset' instead of creating a new one.


Optimal tuning batch size found: 15564
Saved tuning batch size 15564 for AMLSim_MLP
Resuming optimization for MLP on AMLSim: 3 trials done, 147 remaining (target: 150).


[I 2026-05-20 13:47:12,880] Trial 3 finished with value: 0.21124864000366123 and parameters: {'hidden_units': 67, 'dropout_1': 0.29922866146248184, 'dropout_2': 0.1815804216041209, 'learning_rate': 0.002269476700928006, 'weight_decay': 9.784668452846735e-05, 'gamma_focal': 0.7454235172701266, 'n_epochs': 219}. Best is trial 3 with value: 0.21124864000366123.
[I 2026-05-20 13:47:53,750] Trial 4 finished with value: 0.21768083240463187 and parameters: {'hidden_units': 42, 'dropout_1': 0.19535568544465892, 'dropout_2': 0.6036018084477521, 'learning_rate': 0.0027697729101626065, 'weight_decay': 0.00021549210777661239, 'gamma_focal': 1.7423035917688754, 'n_epochs': 150}. Best is trial 4 with value: 0.21768083240463187.
[I 2026-05-20 13:48:33,140] Trial 5 finished with value: 0.219569375483523 and parameters: {'hidden_units': 82, 'dropout_1': 0.6574957116328851, 'dropout_2': 0.08981170547841556, 'learning_rate': 0.010531351573916752, 'weight_decay': 0.0027741541808178395, 'gamma_focal': 2.79

Best val PR-AUC for MLP on AMLSim: 0.2398
GPU memory cleared after AMLSim

✓ Successfully completed tuning for dataset 1/1: AMLSim


HYPERPARAMETER TUNING COMPLETE
All 1 datasets have been processed.


In [ ]:
for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Final evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    # Load best parameters from Optuna databases
    model_parameters = {}
    all_found = True
    for model_name in models:
        study_name = f'{model_name}_optimization on {dataset} dataset'
        storage_url = f'sqlite:///optimization_results_on_{dataset}_{model_name}.db'
        try:
            study = optuna.load_study(study_name=study_name, storage=storage_url)
            model_parameters[model_name] = [study.best_trial.params]
            print(f"Loaded best params for {model_name} (best F1-illicit: {study.best_value:.4f})")
        except KeyError:
            print(f"No completed study found for {model_name} on {dataset}, skipping.")
            all_found = False

    if not model_parameters:
        print(f"No model parameters found for {dataset}. Skipping evaluation.")
        continue

    # Load dataset
    match dataset:
        case "Elliptic":
            from pre_processing import EllipticDataset
            data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
        case "IBM_AML_HiSmall":
            from pre_processing import IBMAMLDataset_HiSmall
            data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
        case "IBM_AML_LiSmall":
            from pre_processing import IBMAMLDataset_LiSmall
            data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
        case "IBM_AML_HiMedium":
            from pre_processing import IBMAMLDataset_HiMedium
            data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
        case "IBM_AML_LiMedium":
            from pre_processing import IBMAMLDataset_LiMedium
            data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
        case "AMLSim":
            from pre_processing import AMLSimDataset
            data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]

    data, masks = extract_and_remove_masks(data)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for evaluation, so all datasets except Elliptic are in this set.

    if dataset not in loader_datasets:
        data = data.to(device)
        print(f"Data moved to device: {device}")
    else:
        print(f"Data kept on CPU for NeighborLoader-based evaluation ({dataset})")

    run_final_evaluation(
        models=list(model_parameters.keys()),
        model_parameters=model_parameters,
        data=data,
        data_for_optimisation=dataset,
        masks=masks
    )

    # Clean up GPU memory between datasets
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f"GPU memory cleared after {dataset}")

    print(f"\n{'='*80}")
    print(f"✓ Successfully completed evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"ALL PROCESSING COMPLETE")
print(f"{'='*80}")


Final evaluation for dataset 1/1: AMLSim

Loaded best params for MLP (best F1-illicit: 0.2398)
Data kept on CPU for NeighborLoader-based evaluation (AMLSim)

Starting FINAL EVALUATION for AMLSim dataset...
Evaluating MLP on AMLSim...
Finding optimal EVALUATION batch size for MLP...
Searching for optimal evaluation batch size...
Running in EVALUATION mode: will use up to 80% of GPU memory
GPU memory limit: 12.79 GB


c:\Users\lambe\Desktop\Final_script\pre_processing.py:766: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Fallback testing batch size: 16384
Optimal evaluation batch size found: 15892
Saved evaluation batch size 15892 for AMLSim_MLP
  > Using EVALUATION batch size 15892 for MLP
Loaded cached tuning batch size 15564 for AMLSim_MLP
  > Using TUNING batch size 15564 for MLP
  > Run 1/10 (Seed 3910815492)
Saved metrics to results/AMLSim/pkl_files/MLP_run_1_pr_data_local-39408.pkl
Saved metrics to results/AMLSim/pkl_files/MLP_run_1_metrics_local-39408.pkl
  > Run 2/10 (Seed 3956897681)
Saved metrics to results/AMLSim/pkl_files/MLP_run_2_pr_data_local-39408.pkl
Saved metrics to results/AMLSim/pkl_files/MLP_run_2_metrics_local-39408.pkl
  > Run 3/10 (Seed 2634484418)
Saved metrics to results/AMLSim/pkl_files/MLP_run_3_pr_data_local-39408.pkl
Saved metrics to results/AMLSim/pkl_files/MLP_run_3_metrics_local-39408.pkl
  > Run 4/10 (Seed 2398472789)
Saved metrics to results/AMLSim/pkl_files/MLP_run_4_pr_data_local-39408.pkl
Saved metrics to results/AMLSim/pkl_files/MLP_run_4_metrics_local-39408.pkl


Restarted Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["IBM_AML_HiSmall", "IBM_AML_LiSmall"]
    models = [ "MLP", "GCN", "GAT", "GIN", "XGB", "RF", "SVM"]
#"IBM_AML_HiSmall", "IBM_AML_LiSmall", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "AMLSim", "Elliptic"
print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 19.5 GB available, 37.3% used
Memory limit: none detected (using host RAM)
--- cgroup diagnostic (probe returned None) ---
  (failed to read /proc/self/cgroup: [Errno 2] No such file or directory: '/proc/self/cgroup')
  (failed to read /proc/self/mountinfo: [Errno 2] No such file or directory: '/proc/self/mountinfo')
ls /sys/fs/cgroup/memory: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup/memory')
ls /sys/fs/cgroup: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup')
PBS/MEM env vars: {}
--- end diagnostic ---
Starting batch processing for 2 datasets: IBM_AML_HiSmall, IBM_AML_LiSmall

Hyperparameter tuning for datasets 1/2: IBM_AML_HiSmall

Study 'MLP_optimization on IBM_AML_HiSmall dataset' not found.


c:\Users\lambe\Desktop\Final_script\pre_processing.py:99: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Dataset IBM_AML_HiSmall loaded successfully for hyperparameter tuning.
Masks extracted and removed from data variable
Starting hyperparameter optimization for IBM_AML_HiSmall dataset...
Data kept on CPU for NeighborLoader-based training (IBM_AML_HiSmall)
Starting hyperparameter optimization for IBM_AML_HiSmall dataset...
Starting hyperparameter tuning for dataset: IBM_AML_HiSmall


Models:   0%|          | 0/7 [00:00<?, ?model/s]

...

[I 2026-05-20 15:18:46,603] A new study created in RDB with name: MLP_optimization on IBM_AML_HiSmall dataset


Batch size 1800192 failed with GPU OOM/CUDA error: CUDA out of memory. Tried to allocate 1.93 GiB. GPU 0 has a total capacity of 15.99 GiB of which 4.57 GiB is free. Of the allocated memory 10.06 GiB is allocated by PyTorch, and 25.44 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Optimal tuning batch size found: 1708236
Saved tuning batch size 1708236 for IBM_AML_HiSmall_MLP
Starting optimization for MLP on IBM_AML_HiSmall with 150 trials.


[I 2026-05-20 15:29:02,810] Trial 0 finished with value: 0.012810492783294172 and parameters: {'hidden_units': 106, 'dropout_1': 0.17871756264244057, 'dropout_2': 0.20329499780325447, 'learning_rate': 0.00021652270628788215, 'weight_decay': 0.0007174160121620881, 'gamma_focal': 1.9773041395294226, 'n_epochs': 237}. Best is trial 0 with value: 0.012810492783294172.
[I 2026-05-20 15:38:01,612] Trial 1 pruned. 
[I 2026-05-20 15:47:36,834] Trial 2 finished with value: 0.012097230343653988 and parameters: {'hidden_units': 114, 'dropout_1': 0.21118888780478748, 'dropout_2': 0.09354344852708302, 'learning_rate': 0.0001984005926946891, 'weight_decay': 6.455999407813551e-05, 'gamma_focal': 1.4322693003479183, 'n_epochs': 224}. Best is trial 0 with value: 0.012810492783294172.
[I 2026-05-20 15:52:35,572] Trial 3 finished with value: 0.011518570783598765 and parameters: {'hidden_units': 35, 'dropout_1': 0.07305427811095758, 'dropout_2': 0.6885432512299886, 'learning_rate': 0.0006694077796661983, 

CUDA OOM at epoch 1. Pruning trial.
CUDA OOM at epoch 1. Pruning trial.


[W 2026-05-20 15:58:14,672] Trial 7 failed with parameters: {'hidden_units': 95, 'dropout_1': 0.2188040211788521, 'dropout_2': 0.07180770946610458, 'learning_rate': 0.00037346673168626196, 'weight_decay': 0.0002500223727325357, 'gamma_focal': 3.7991350483863053, 'n_epochs': 221} because of the following error: OutOfMemoryError('CUDA out of memory. Tried to allocate 706.00 MiB. GPU 0 has a total capacity of 15.99 GiB of which 1.98 GiB is free. Of the allocated memory 11.85 GiB is allocated by PyTorch, and 37.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)').
Traceback (most recent call last):
  File "c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\optuna\study\_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  Fi

In [ ]:
for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Final evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    # Load best parameters from Optuna databases
    model_parameters = {}
    all_found = True
    for model_name in models:
        study_name = f'{model_name}_optimization on {dataset} dataset'
        storage_url = f'sqlite:///optimization_results_on_{dataset}_{model_name}.db'
        try:
            study = optuna.load_study(study_name=study_name, storage=storage_url)
            model_parameters[model_name] = [study.best_trial.params]
            print(f"Loaded best params for {model_name} (best F1-illicit: {study.best_value:.4f})")
        except KeyError:
            print(f"No completed study found for {model_name} on {dataset}, skipping.")
            all_found = False

    if not model_parameters:
        print(f"No model parameters found for {dataset}. Skipping evaluation.")
        continue

    # Load dataset
    match dataset:
        case "Elliptic":
            from pre_processing import EllipticDataset
            data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
        case "IBM_AML_HiSmall":
            from pre_processing import IBMAMLDataset_HiSmall
            data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
        case "IBM_AML_LiSmall":
            from pre_processing import IBMAMLDataset_LiSmall
            data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
        case "IBM_AML_HiMedium":
            from pre_processing import IBMAMLDataset_HiMedium
            data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
        case "IBM_AML_LiMedium":
            from pre_processing import IBMAMLDataset_LiMedium
            data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
        case "AMLSim":
            from pre_processing import AMLSimDataset
            data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]

    data, masks = extract_and_remove_masks(data)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for evaluation, so all datasets except Elliptic are in this set.

    if dataset not in loader_datasets:
        data = data.to(device)
        print(f"Data moved to device: {device}")
    else:
        print(f"Data kept on CPU for NeighborLoader-based evaluation ({dataset})")

    run_final_evaluation(
        models=list(model_parameters.keys()),
        model_parameters=model_parameters,
        data=data,
        data_for_optimisation=dataset,
        masks=masks
    )

    # Clean up GPU memory between datasets
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f"GPU memory cleared after {dataset}")

    print(f"\n{'='*80}")
    print(f"✓ Successfully completed evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"ALL PROCESSING COMPLETE")
print(f"{'='*80}")

NameError: name 'datasets' is not defined

Restarted Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["IBM_AML_HiSmall", "IBM_AML_LiSmall"]
    models = [ "MLP", "GCN", "GAT", "GIN", "XGB", "RF", "SVM"]
#"IBM_AML_HiSmall", "IBM_AML_LiSmall", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "AMLSim", "Elliptic"
print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 17.7 GB available, 43.2% used
Memory limit: none detected (using host RAM)
--- cgroup diagnostic (probe returned None) ---
  (failed to read /proc/self/cgroup: [Errno 2] No such file or directory: '/proc/self/cgroup')
  (failed to read /proc/self/mountinfo: [Errno 2] No such file or directory: '/proc/self/mountinfo')
ls /sys/fs/cgroup/memory: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup/memory')
ls /sys/fs/cgroup: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup')
PBS/MEM env vars: {}
--- end diagnostic ---
Starting batch processing for 2 datasets: IBM_AML_HiSmall, IBM_AML_LiSmall

Hyperparameter tuning for datasets 1/2: IBM_AML_HiSmall

Study 'MLP_optimization on IBM_AML_HiSmall dataset' incomplete: 8/150 trials done, 142 remaining.


c:\Users\lambe\Desktop\Final_script\pre_processing.py:99: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Dataset IBM_AML_HiSmall loaded successfully for hyperparameter tuning.
Masks extracted and removed from data variable
Starting hyperparameter optimization for IBM_AML_HiSmall dataset...
Data kept on CPU for NeighborLoader-based training (IBM_AML_HiSmall)
Starting hyperparameter optimization for IBM_AML_HiSmall dataset...
Starting hyperparameter tuning for dataset: IBM_AML_HiSmall


Models:   0%|          | 0/7 [00:00<?, ?model/s][I 2026-05-20 16:04:15,551] Using an existing study with name 'MLP_optimization on IBM_AML_HiSmall dataset' instead of creating a new one.


Study 'MLP_optimization on IBM_AML_HiSmall dataset' incomplete: 8/150 trials done, 142 remaining.
Resuming optimization for MLP on IBM_AML_HiSmall: 8 trials done, 142 remaining (target: 150).


[W 2026-05-20 16:04:16,073] Trial 9 failed with parameters: {'hidden_units': 190, 'dropout_1': 0.5614033467789953, 'dropout_2': 0.33257933549785856, 'learning_rate': 0.010566639347773245, 'weight_decay': 0.0036963330483027633, 'gamma_focal': 1.88017434796407, 'n_epochs': 180} because of the following error: RuntimeError('indices should be either on cpu or on the same device as the indexed tensor (cpu)').
Traceback (most recent call last):
  File "c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\optuna\study\_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "c:\Users\lambe\Desktop\Final_script\funcs_for_optuna.py", line 134, in <lambda>
    lambda trial: run_trial_with_cleanup(
  File "c:\Users\lambe\Desktop\Final_script\helper_functions.py", line 369, in run_trial_with_cleanup
    result = trial_func(*args, **kwargs)
  File "c:\Users\lambe\Desktop\Final_script\funcs_for_optuna.py", line 201, in objective
    x_train = data.x[train_mask_dev].to(

RuntimeError: indices should be either on cpu or on the same device as the indexed tensor (cpu)

In [ ]:
for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Final evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    # Load best parameters from Optuna databases
    model_parameters = {}
    all_found = True
    for model_name in models:
        study_name = f'{model_name}_optimization on {dataset} dataset'
        storage_url = f'sqlite:///optimization_results_on_{dataset}_{model_name}.db'
        try:
            study = optuna.load_study(study_name=study_name, storage=storage_url)
            model_parameters[model_name] = [study.best_trial.params]
            print(f"Loaded best params for {model_name} (best F1-illicit: {study.best_value:.4f})")
        except KeyError:
            print(f"No completed study found for {model_name} on {dataset}, skipping.")
            all_found = False

    if not model_parameters:
        print(f"No model parameters found for {dataset}. Skipping evaluation.")
        continue

    # Load dataset
    match dataset:
        case "Elliptic":
            from pre_processing import EllipticDataset
            data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
        case "IBM_AML_HiSmall":
            from pre_processing import IBMAMLDataset_HiSmall
            data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
        case "IBM_AML_LiSmall":
            from pre_processing import IBMAMLDataset_LiSmall
            data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
        case "IBM_AML_HiMedium":
            from pre_processing import IBMAMLDataset_HiMedium
            data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
        case "IBM_AML_LiMedium":
            from pre_processing import IBMAMLDataset_LiMedium
            data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
        case "AMLSim":
            from pre_processing import AMLSimDataset
            data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]

    data, masks = extract_and_remove_masks(data)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for evaluation, so all datasets except Elliptic are in this set.

    if dataset not in loader_datasets:
        data = data.to(device)
        print(f"Data moved to device: {device}")
    else:
        print(f"Data kept on CPU for NeighborLoader-based evaluation ({dataset})")

    run_final_evaluation(
        models=list(model_parameters.keys()),
        model_parameters=model_parameters,
        data=data,
        data_for_optimisation=dataset,
        masks=masks
    )

    # Clean up GPU memory between datasets
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f"GPU memory cleared after {dataset}")

    print(f"\n{'='*80}")
    print(f"✓ Successfully completed evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"ALL PROCESSING COMPLETE")
print(f"{'='*80}")

: 

Restarted Python

In [ ]:
import os
# Must be set BEFORE any torch/CUDA imports to take effect
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import gc
from dependencies import *
from utilities import *
from helper_functions import check_study_existence
from funcs_for_optuna import hyperparameter_tuning
from testing import run_final_evaluation
import optuna
configure_gpu_memory_limits(fraction=0.75)
seeded_run = False
if seeded_run:
    set_seed(42)
    print("Seeded run with seed 42")
else:
    seed = np.random.SeedSequence().generate_state(1)[0]
    set_seed(seed)
#%% Hyperparameter tuning

#Checking whether the code is running on the HPC or on my windows pc
import platform
if platform.system() == 'Linux':
    # Store all paths when running linux in a variable to be accessed later
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    import sys
    datasets = [sys.argv[1]] #Getting dataset variable from submit.sh script
    models = [sys.argv[2]] #Getting model variable from submit.sh script
    
else:
    dataset_paths = {
        "Elliptic": 'Datasets/Elliptic_dataset',
        "IBM_AML_HiSmall": 'Datasets/IBM_AML_dataset/HiSmall',
        "IBM_AML_LiSmall": 'Datasets/IBM_AML_dataset/LiSmall',
        "IBM_AML_HiMedium": 'Datasets/IBM_AML_dataset/HiMedium',
        "IBM_AML_LiMedium": 'Datasets/IBM_AML_dataset/LiMedium',
        "AMLSim": 'Datasets/AMLSim_dataset'
    }
    datasets = ["IBM_AML_HiSmall", "IBM_AML_LiSmall"]
    models = [ "MLP", "GCN", "GAT", "GIN", "XGB", "RF", "SVM"]
#"IBM_AML_HiSmall", "IBM_AML_LiSmall", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "AMLSim", "Elliptic"
print(f"Starting batch processing for {len(datasets)} datasets: {', '.join(datasets)}")
print("=" * 80)

for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Hyperparameter tuning for datasets {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    #Check if there is a missing study in the database
    if all(check_study_existence(model, dataset) for model in models):
        print(f"All studies for {dataset} are complete. Skipping to next dataset.")
        continue
    else:
        match dataset:
            case "Elliptic":
                from pre_processing import EllipticDataset
                data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
                dataset = "Elliptic"
            case "IBM_AML_HiSmall":
                from pre_processing import IBMAMLDataset_HiSmall
                data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
                dataset = "IBM_AML_HiSmall"
            case "IBM_AML_LiSmall":
                from pre_processing import IBMAMLDataset_LiSmall
                data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
                dataset = "IBM_AML_LiSmall"
            case "IBM_AML_HiMedium":
                from pre_processing import IBMAMLDataset_HiMedium
                data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
                dataset = "IBM_AML_HiMedium"
            case "IBM_AML_LiMedium":
                from pre_processing import IBMAMLDataset_LiMedium
                data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
                dataset = "IBM_AML_LiMedium"
            case "AMLSim":
                from pre_processing import AMLSimDataset
                data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]
                dataset = "AMLSim"
        print(f"Dataset {dataset} loaded successfully for hyperparameter tuning.")

        data, masks = extract_and_remove_masks(data)
        print(f"Masks extracted and removed from data variable")

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for training, so all datasets except Elliptic are in this set.

        #Starting optimisation
        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        # Move data to device only for full-batch datasets; NeighborLoader datasets
        # stay on CPU so the full graph doesn't consume GPU memory
        if dataset not in loader_datasets:
            data = data.to(device)
            print(f"Data moved to device: {device}")
        else:
            print(f"Data kept on CPU for NeighborLoader-based training ({dataset})")

        print(f"Starting hyperparameter optimization for {dataset} dataset...")

        model_parameters = hyperparameter_tuning(
            models=models,
            dataset_name=dataset,
            data=data,
            masks=masks
        )

        # Clean up tuning artifacts before evaluation
        del data, masks, model_parameters
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
        gc.collect()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            print(f"GPU memory cleared after {dataset}")

        print(f"\n{'='*80}")
        print(f"✓ Successfully completed tuning for dataset {idx}/{len(datasets)}: {dataset}")
        print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"HYPERPARAMETER TUNING COMPLETE")
print(f"All {len(datasets)} datasets have been processed.")
print(f"{'='*80}")

c:\Users\lambe\anaconda3\envs\Home_PC_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU memory limited to 75.0% of available memory
PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:512
System RAM: 31.1 GB total, 17.8 GB available, 42.9% used
Memory limit: none detected (using host RAM)
--- cgroup diagnostic (probe returned None) ---
  (failed to read /proc/self/cgroup: [Errno 2] No such file or directory: '/proc/self/cgroup')
  (failed to read /proc/self/mountinfo: [Errno 2] No such file or directory: '/proc/self/mountinfo')
ls /sys/fs/cgroup/memory: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup/memory')
ls /sys/fs/cgroup: ([WinError 3] The system cannot find the path specified: '/sys/fs/cgroup')
PBS/MEM env vars: {}
--- end diagnostic ---
Starting batch processing for 2 datasets: IBM_AML_HiSmall, IBM_AML_LiSmall

Hyperparameter tuning for datasets 1/2: IBM_AML_HiSmall

Study 'MLP_optimization on IBM_AML_HiSmall dataset' incomplete: 9/150 trials done, 141 remaining.


c:\Users\lambe\Desktop\Final_script\pre_processing.py:99: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_paths[0])


Dataset IBM_AML_HiSmall loaded successfully for hyperparameter tuning.
Masks extracted and removed from data variable
Starting hyperparameter optimization for IBM_AML_HiSmall dataset...
Data kept on CPU for NeighborLoader-based training (IBM_AML_HiSmall)
Starting hyperparameter optimization for IBM_AML_HiSmall dataset...
Starting hyperparameter tuning for dataset: IBM_AML_HiSmall


Models:   0%|          | 0/7 [00:00<?, ?model/s][I 2026-05-20 16:07:48,661] Using an existing study with name 'MLP_optimization on IBM_AML_HiSmall dataset' instead of creating a new one.


...

Best val PR-AUC for MLP on IBM_AML_LiSmall: 0.0059
Study 'GCN_optimization on IBM_AML_LiSmall dataset' not found.
Searching for optimal tuning batch size...
Running in TUNING mode: will use up to 70% of GPU memory
GPU memory limit: 11.19 GB
Testing batch size: 16384
Testing batch size: 32768
Testing batch size: 65536
Testing batch size: 131072
Testing batch size: 262144
Testing batch size: 524288
Batch size 524288 failed with GPU OOM/CUDA error: CUDA out of memory. Tried to allocate 892.00 MiB. GPU 0 has a total capacity of 15.99 GiB of which 2.73 GiB is free. Of the allocated memory 11.62 GiB is allocated by PyTorch, and 39.24 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
Binary search testing: 393216
Binary search testing: 458752
Binary search tes

[I 2026-05-22 06:37:45,827] A new study created in RDB with name: GCN_optimization on IBM_AML_LiSmall dataset


Optimal tuning batch size found: 464998
Saved tuning batch size 464998 for IBM_AML_LiSmall_GCN
Starting optimization for GCN on IBM_AML_LiSmall with 150 trials.


[I 2026-05-22 06:39:08,212] Trial 0 finished with value: 0.004295725156052315 and parameters: {'hidden_units': 158, 'dropout': 0.021838737409128325, 'n_layers': 2, 'learning_rate': 0.008280412973841923, 'weight_decay': 0.00013403355545454538, 'gamma_focal': 1.1637400534186046, 'n_epochs': 16}. Best is trial 0 with value: 0.004295725156052315.
[I 2026-05-22 06:51:00,371] Trial 1 finished with value: 0.003553840557983409 and parameters: {'hidden_units': 60, 'dropout': 0.36696062313690037, 'n_layers': 1, 'learning_rate': 0.03803495077849978, 'weight_decay': 0.00456983588543006, 'gamma_focal': 0.65778269385443, 'n_epochs': 150}. Best is trial 0 with value: 0.004295725156052315.


: 

In [ ]:
for idx, dataset in enumerate(datasets, 1):
    print(f"\n{'='*80}")
    print(f"Final evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

    # Load best parameters from Optuna databases
    model_parameters = {}
    all_found = True
    for model_name in models:
        study_name = f'{model_name}_optimization on {dataset} dataset'
        storage_url = f'sqlite:///optimization_results_on_{dataset}_{model_name}.db'
        try:
            study = optuna.load_study(study_name=study_name, storage=storage_url)
            model_parameters[model_name] = [study.best_trial.params]
            print(f"Loaded best params for {model_name} (best F1-illicit: {study.best_value:.4f})")
        except KeyError:
            print(f"No completed study found for {model_name} on {dataset}, skipping.")
            all_found = False

    if not model_parameters:
        print(f"No model parameters found for {dataset}. Skipping evaluation.")
        continue

    # Load dataset
    match dataset:
        case "Elliptic":
            from pre_processing import EllipticDataset
            data = EllipticDataset(root=dataset_paths["Elliptic"])[0]
        case "IBM_AML_HiSmall":
            from pre_processing import IBMAMLDataset_HiSmall
            data = IBMAMLDataset_HiSmall(root=dataset_paths["IBM_AML_HiSmall"])[0]
        case "IBM_AML_LiSmall":
            from pre_processing import IBMAMLDataset_LiSmall
            data = IBMAMLDataset_LiSmall(root=dataset_paths["IBM_AML_LiSmall"])[0]
        case "IBM_AML_HiMedium":
            from pre_processing import IBMAMLDataset_HiMedium
            data = IBMAMLDataset_HiMedium(root=dataset_paths["IBM_AML_HiMedium"])[0]
        case "IBM_AML_LiMedium":
            from pre_processing import IBMAMLDataset_LiMedium
            data = IBMAMLDataset_LiMedium(root=dataset_paths["IBM_AML_LiMedium"])[0]
        case "AMLSim":
            from pre_processing import AMLSimDataset
            data = AMLSimDataset(root=dataset_paths["AMLSim"])[0]

    data, masks = extract_and_remove_masks(data)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader_datasets = {"AMLSim", "IBM_AML_HiMedium", "IBM_AML_LiMedium", "IBM_AML_HiSmall", "IBM_AML_LiSmall"} #All datasets except Elliptic use NeighborLoader for evaluation, so all datasets except Elliptic are in this set.

    if dataset not in loader_datasets:
        data = data.to(device)
        print(f"Data moved to device: {device}")
    else:
        print(f"Data kept on CPU for NeighborLoader-based evaluation ({dataset})")

    run_final_evaluation(
        models=list(model_parameters.keys()),
        model_parameters=model_parameters,
        data=data,
        data_for_optimisation=dataset,
        masks=masks
    )

    # Clean up GPU memory between datasets
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f"GPU memory cleared after {dataset}")

    print(f"\n{'='*80}")
    print(f"✓ Successfully completed evaluation for dataset {idx}/{len(datasets)}: {dataset}")
    print(f"{'='*80}\n")

print(f"\n{'='*80}")
print(f"ALL PROCESSING COMPLETE")
print(f"{'='*80}")